# 00 - Build allocation input database

This notebook prepares allocation lookup/resource tables from source files in `data/`, then creates `data/allocation_model.db`.

Run this when starting from source files, or when one of the allocation inputs has changed.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in ["notebooks", "allocation"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ALLOC_DB_PATH = DATA_DIR / "allocation_model.db"
POLICE_DB_PATH = DATA_DIR / "police_data.db"
WALES_DB_PATH = DATA_DIR / "wales_data.db"

DATA_DIR.mkdir(exist_ok=True)

source_files = {
    "english_predictions": DATA_DIR / "english_predictions.csv",
    "welsh_predictions": DATA_DIR / "welsh_predictions.csv",
    "postcode_lookup": DATA_DIR / "PCD_OA21_LSOA21_MSOA21_LAD_MAY25_UK_LU.csv",
    "rurality_lookup": DATA_DIR / "Rural_Urban_Classification_(2021)_of_LSOAs_in_EW.csv",
    "msoa_population": DATA_DIR / "sapemsoaquinaryage20222024.xlsx",
    "police_workforce": DATA_DIR / "open-data-table-police-workforce-functions-280126.ods",
}

missing = [str(path) for path in source_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing source files: " + ", ".join(missing))

for db_path in [POLICE_DB_PATH, WALES_DB_PATH]:
    if not db_path.exists():
        raise FileNotFoundError(f"Missing database: {db_path}")

print("source files:")
for name, path in source_files.items():
    print(f"- {name}: {path}")
print("allocation database will be rebuilt at:", ALLOC_DB_PATH)

## Prepare input CSVs from source files

In [ ]:
forecast_columns = [
    "lsoa_code",
    "crime_type",
    "month",
    "horizon_step",
    "q0.02",
    "q0.10",
    "q0.25",
    "q0.50",
    "q0.75",
    "q0.90",
    "q0.98",
]

forecast_predictions_england = pd.read_csv(source_files["english_predictions"])[forecast_columns]
forecast_predictions_wales = pd.read_csv(source_files["welsh_predictions"])[forecast_columns]
forecast_predictions = pd.concat(
    [forecast_predictions_england, forecast_predictions_wales],
    ignore_index=True,
)

lsoa_msoa_lookup = (
    pd.read_csv(
        source_files["postcode_lookup"],
        usecols=["lsoa21cd", "msoa21cd", "lsoa21nm", "msoa21nm"],
        dtype=str,
    )
    .rename(
        columns={
            "lsoa21cd": "lsoa_code",
            "msoa21cd": "msoa_code",
            "lsoa21nm": "lsoa_name_lookup",
            "msoa21nm": "msoa_name",
        }
    )
    .dropna(subset=["lsoa_code", "msoa_code"])
    .drop_duplicates()
    .reset_index(drop=True)
)

lsoa_rurality = (
    pd.read_csv(source_files["rurality_lookup"])
    .rename(
        columns={
            "LSOA21CD": "lsoa_code",
            "RUC21NM": "ruc_name",
            "Urban_rural_flag": "urban_rural_flag",
        }
    )[["lsoa_code", "ruc_name", "urban_rural_flag"]]
)
lsoa_rurality["is_rural"] = (
    lsoa_rurality["urban_rural_flag"].str.lower() == "rural"
).astype(int)

msoa_population = pd.read_excel(
    source_files["msoa_population"],
    sheet_name="Mid-2024 MSOA 2021",
    header=3,
    engine="openpyxl",
)[
    ["LAD 2023 Code", "LAD 2023 Name", "MSOA 2021 Code", "MSOA 2021 Name", "Total"]
].rename(
    columns={
        "LAD 2023 Code": "lad_code",
        "LAD 2023 Name": "lad_name",
        "MSOA 2021 Code": "msoa_code",
        "MSOA 2021 Name": "msoa_name",
        "Total": "population_2024",
    }
)
msoa_population["population_2024"] = msoa_population["population_2024"].astype(int)

workforce_columns = {
    "As at 31 March": "year",
    "Geocode": "pfa_code",
    "Force name": "pfa_name",
    "Region": "region",
    "Worker type": "worker_type",
    "Ethnicity (5+1)": "ethnicity_5_1",
    "Ethnicity (3+1)": "ethnicity_3_1",
    "Sex": "sex",
    "Function subgroup number": "function_subgroup_number",
    "Function subgroup name": "function_subgroup_name",
    "Wider function number": "wider_function_number",
    "Wider function name": "wider_function_name",
    "Frontline type": "frontline_type",
    "Total (FTE)": "fte",
}

police_workforce = pd.read_excel(
    source_files["police_workforce"],
    sheet_name="Data",
    engine="odf",
    dtype=str,
).rename(columns=workforce_columns)[list(workforce_columns.values())]

police_workforce["year"] = police_workforce["year"].astype(int)
police_workforce["fte"] = pd.to_numeric(
    police_workforce["fte"]
    .str.replace(" ", "", regex=False)
    .str.replace(",", "", regex=False)
    .replace({"[z]": None, "x": None, "-": None, "": None}),
    errors="coerce",
)

print("prepared rows")
print("england predictions:", len(forecast_predictions_england))
print("wales predictions:", len(forecast_predictions_wales))
print("combined predictions:", len(forecast_predictions))
print("lsoa to msoa lookup:", len(lsoa_msoa_lookup))
print("msoa population:", len(msoa_population))
print("lsoa rurality:", len(lsoa_rurality))
print("police workforce:", len(police_workforce))

## Build MSOA To Police Force Lookup

In [ ]:
def read_lsoa_info(db_path):
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(
            "SELECT lsoa_code, pfa_code, pfa_name FROM lsoa_info;",
            conn,
        )

lsoa_info = pd.concat(
    [read_lsoa_info(POLICE_DB_PATH), read_lsoa_info(WALES_DB_PATH)],
    ignore_index=True,
)

msoa_force_lookup = (
    lsoa_info
    .merge(lsoa_msoa_lookup[["lsoa_code", "msoa_code"]], on="lsoa_code", how="inner")
    [["msoa_code", "pfa_code", "pfa_name"]]
    .drop_duplicates()
    .sort_values(["msoa_code", "pfa_code", "pfa_name"])
    .reset_index(drop=True)
)

print("msoa force lookup:", len(msoa_force_lookup))
msoa_force_lookup.head()

## Save Fresh allocation_model.db

In [ ]:
if ALLOC_DB_PATH.exists():
    ALLOC_DB_PATH.unlink()

with sqlite3.connect(ALLOC_DB_PATH) as conn:
    forecast_predictions_england.to_sql("forecast_predictions_england", conn, if_exists="replace", index=False)
    forecast_predictions_wales.to_sql("forecast_predictions_wales", conn, if_exists="replace", index=False)
    forecast_predictions.to_sql("forecast_predictions", conn, if_exists="replace", index=False)
    lsoa_msoa_lookup.to_sql("lsoa_msoa_lookup", conn, if_exists="replace", index=False)
    msoa_force_lookup.to_sql("msoa_force_lookup", conn, if_exists="replace", index=False)
    msoa_population.to_sql("msoa_population_2024", conn, if_exists="replace", index=False)
    lsoa_rurality.to_sql("lsoa_rurality", conn, if_exists="replace", index=False)
    police_workforce.to_sql("police_workforce_resources", conn, if_exists="replace", index=False)

input_summary = pd.DataFrame(
    [
        {
            "input": "english_predictions.csv",
            "rows": len(forecast_predictions_england),
            "distinct_lsoas": forecast_predictions_england["lsoa_code"].nunique(),
        },
        {
            "input": "welsh_predictions.csv",
            "rows": len(forecast_predictions_wales),
            "distinct_lsoas": forecast_predictions_wales["lsoa_code"].nunique(),
        },
        {
            "input": "combined forecast_predictions",
            "rows": len(forecast_predictions),
            "distinct_lsoas": forecast_predictions["lsoa_code"].nunique(),
        },
        {
            "input": "msoa_population_2024.csv",
            "rows": len(msoa_population),
            "distinct_lsoas": None,
        },
        {
            "input": "msoa_force_lookup",
            "rows": len(msoa_force_lookup),
            "distinct_lsoas": None,
        },
    ]
)

print("rebuilt", ALLOC_DB_PATH)
input_summary